In [ ]:
# =============================================================
# 08_task_D_mnar_imputation.ipynb
# Task D — MNAR Imputation Benchmark
# ------------------------------------------------------------
# Benchmark task: impute structural missing values in the
# panel and evaluate quality under the income-gradient MNAR
# mechanism described in 4.1.
#
# MNAR mechanism (4.1)
# ─────────────────────
# Gini index, income share (low 20%), and poverty headcount
# are structurally missing for low-income and lower-middle-
# income countries (coverage ≤ 40%). Missingness correlates
# with income level — countries with highest inequality
# tend to have the least inequality data. This is the
# canonical MNAR (Missing Not At Random) condition: the
# probability of missingness depends on the unobserved value.
#
# Evaluation protocol
# ────────────────────
# Amputation experiment:
#   1. Take complete-case rows for the three inequality
#      indicators (countries with observed Gini data).
#   2. Artificially introduce MAR/MNAR patterns by masking
#      a fraction of observed values.
#   3. Impute using each method.
#   4. Evaluate: RMSE and MAE vs original observed values.
#
# Methods compared
# ─────────────────
#   1. Mean imputation (column mean, ignores structure)
#   2. Median imputation (column median)
#   3. Forward-fill (last observed value, temporal)
#   4. Linear interpolation (within country)
#   5. KNN imputation (k=5, cross-country similarity)
#   6. MICE / IterativeImputer (multivariate, BayesianRidge)
#
# Additionally, for the STRUCTURAL MNAR gap
# (countries with ALL Gini values missing):
#   7. Income-group mean imputation
#      (mean Gini within same World Bank income tier)
#   8. GDP-anchored regression imputation
#      (OLS: Gini ~ log(GDP/capita) across observed countries)
#
# Outputs
# ───────
#   outputs/results/task_D_metrics.csv
#   outputs/tables/table_taskD.csv / .tex
#   outputs/figures/panel_12/taskD_imputation_rmse.png
#   outputs/figures/panel_12/taskD_mnar_pattern.png
#   outputs/figures/panel_12/taskD_imputed_gini.png
#
# Usage
# ─────
#   python src/08_task_D_mnar_imputation.py
#
# Requirements
# ────────────
#   pip install scikit-learn pandas numpy matplotlib pyyaml
# =============================================================

from __future__ import annotations

import sys
import warnings
from pathlib import Path

try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    _cwd = Path.cwd()
    _SRC = _cwd / "src" if (_cwd / "src").exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge, LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

from utils import find_project_root, load_indicators, load_pipeline, INCOME_ORDER

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

ALL_INDICATORS = list(IND_CFG["indicators"].keys())

# MNAR target indicators — the three structural inequality measures
MNAR_TARGETS = ["gini", "inc_share_low20", "poverty_215"]

# Additional covariates used for model-based imputation
IMPUTE_COVARIATES = [
    "gdp_pc_ppp", "tert_enrol_gross", "sec_completion",
    "gov_effect", "work_age_share",
]

# Amputation settings
AMPUTATE_FRAC  = 0.30   # fraction of observed Gini values to mask for evaluation
RANDOM_SEED    = 42

DATA_DIR   = PROJECT_ROOT / "data" / "processed"
RESULT_DIR = PROJECT_ROOT / "outputs" / "results"
TABLE_DIR  = PROJECT_ROOT / "outputs" / "tables"
FIGDIR     = PROJECT_ROOT / "outputs" / "figures" / "panel_12"

for d in [RESULT_DIR, TABLE_DIR, FIGDIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[info] Project root  : {PROJECT_ROOT}")
print(f"[info] MNAR targets  : {MNAR_TARGETS}")
print(f"[info] Amputation    : {AMPUTATE_FRAC:.0%} of observed values masked")


# ──────────────────────────────────────────────────────────────
# Load data
# ──────────────────────────────────────────────────────────────
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    panel_path = DATA_DIR / f"panel_12countries_{PERIOD_TAG}.csv"
    mask_path  = DATA_DIR / f"interpolation_mask_{PERIOD_TAG}.csv"
    for p in [panel_path, mask_path]:
        if not p.exists():
            raise FileNotFoundError(f"Required: {p}")
    panel = pd.read_csv(panel_path)
    mask  = pd.read_csv(mask_path)
    print(f"\n[step 0] Panel: {len(panel)} rows × {len(panel.columns)} cols")
    return panel, mask


# ──────────────────────────────────────────────────────────────
# Step 1 — Describe MNAR structure
# ──────────────────────────────────────────────────────────────
def describe_mnar(panel: pd.DataFrame) -> pd.DataFrame:
    """
    For each country, compute Gini coverage and income tier.
    Print the income-gradient MNAR pattern.
    """
    records = []
    for iso, grp in panel.groupby("iso3c"):
        for tgt in MNAR_TARGETS:
            if tgt in grp.columns:
                cov  = float(grp[tgt].notna().mean())
                miss = int(grp[tgt].isna().sum())
                records.append({
                    "iso3c":        iso,
                    "country":      grp["country"].iloc[0],
                    "income_group": grp["income_group"].iloc[0],
                    "indicator":    tgt,
                    "coverage":     cov,
                    "n_missing":    miss,
                    "n_observed":   10 - miss,
                })
    df = pd.DataFrame(records)

    print(f"\n[step 1] MNAR pattern — Gini coverage by income tier:")
    gini_df = df[df["indicator"] == "gini"].copy()
    for tier in ["High income", "Upper middle income",
                 "Lower middle income", "Low income"]:
        sub = gini_df[gini_df["income_group"] == tier]
        if sub.empty:
            continue
        mean_cov = sub["coverage"].mean()
        isos     = ", ".join(sub["iso3c"].tolist())
        print(f"  {tier:<25}  mean_cov={mean_cov:.0%}  ({isos})")

    return df


# ──────────────────────────────────────────────────────────────
# Step 2 — Amputation experiment (observed Gini rows)
# ──────────────────────────────────────────────────────────────
def ampute(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    For countries with observed Gini data, mask AMPUTATE_FRAC
    of their observations at random (MAR amputation).

    Returns
    -------
    panel_amp  : panel with masked values set to NaN
    truth_df   : original values at masked positions
    """
    rng       = np.random.default_rng(RANDOM_SEED)
    panel_amp = panel.copy()

    truth_rows = []
    for iso, grp in panel.groupby("iso3c"):
        for tgt in MNAR_TARGETS:
            if tgt not in grp.columns:
                continue
            obs_idx = grp.index[grp[tgt].notna()].tolist()
            if len(obs_idx) < 3:
                continue   # too few to amputate
            n_mask = max(1, int(len(obs_idx) * AMPUTATE_FRAC))
            masked = rng.choice(obs_idx, size=n_mask, replace=False)
            for idx in masked:
                truth_rows.append({
                    "iso3c":     iso,
                    "year":      panel.loc[idx, "year"],
                    "indicator": tgt,
                    "true_value":panel_amp.loc[idx, tgt],
                })
                panel_amp.loc[idx, tgt] = np.nan

    truth_df = pd.DataFrame(truth_rows)
    print(f"\n[step 2] Amputation: {len(truth_df)} values masked "
          f"across {truth_df['iso3c'].nunique()} countries, "
          f"{truth_df['indicator'].nunique()} indicators")
    return panel_amp, truth_df


# ──────────────────────────────────────────────────────────────
# Step 3 — Imputation methods
# ──────────────────────────────────────────────────────────────
def _eval(
    panel_imp:  pd.DataFrame,
    truth_df:   pd.DataFrame,
    method_name: str,
) -> dict:
    """
    Compare imputed values at masked positions against ground truth.
    Returns dict of RMSE and MAE per indicator and overall.
    """
    records = []
    for _, row in truth_df.iterrows():
        iso = row["iso3c"]
        yr  = row["year"]
        tgt = row["indicator"]
        tv  = row["true_value"]
        idx = panel_imp.index[
            (panel_imp["iso3c"] == iso) & (panel_imp["year"] == yr)
        ]
        if len(idx) == 0:
            continue
        iv  = panel_imp.loc[idx[0], tgt]
        records.append({
            "iso3c":      iso,
            "year":       yr,
            "indicator":  tgt,
            "true_value": tv,
            "imp_value":  iv,
        })

    df = pd.DataFrame(records).dropna(subset=["imp_value"])
    if df.empty:
        return {"method": method_name,
                "rmse_overall": np.nan, "mae_overall": np.nan,
                **{f"rmse_{t}": np.nan for t in MNAR_TARGETS},
                **{f"mae_{t}":  np.nan for t in MNAR_TARGETS},
                "n_evaluated": 0}

    rmse_all = float(np.sqrt(mean_squared_error(
        df["true_value"], df["imp_value"])))
    mae_all  = float(mean_absolute_error(
        df["true_value"], df["imp_value"]))

    result = {
        "method": method_name,
        "rmse_overall": round(rmse_all, 4),
        "mae_overall":  round(mae_all, 4),
        "n_evaluated":  len(df),
    }
    for tgt in MNAR_TARGETS:
        sub = df[df["indicator"] == tgt]
        if len(sub) >= 2:
            result[f"rmse_{tgt}"] = round(float(np.sqrt(mean_squared_error(
                sub["true_value"], sub["imp_value"]))), 4)
            result[f"mae_{tgt}"]  = round(float(mean_absolute_error(
                sub["true_value"], sub["imp_value"])), 4)
        else:
            result[f"rmse_{tgt}"] = np.nan
            result[f"mae_{tgt}"]  = np.nan
    return result


def method_mean(panel_amp: pd.DataFrame, truth_df: pd.DataFrame) -> dict:
    p = panel_amp.copy()
    for tgt in MNAR_TARGETS:
        if tgt in p.columns:
            p[tgt] = p[tgt].fillna(p[tgt].mean())
    return _eval(p, truth_df, "Mean")


def method_median(panel_amp: pd.DataFrame, truth_df: pd.DataFrame) -> dict:
    p = panel_amp.copy()
    for tgt in MNAR_TARGETS:
        if tgt in p.columns:
            p[tgt] = p[tgt].fillna(p[tgt].median())
    return _eval(p, truth_df, "Median")


def method_ffill(panel_amp: pd.DataFrame, truth_df: pd.DataFrame) -> dict:
    """Forward-fill within country (temporal carry-forward)."""
    p = panel_amp.copy()
    for tgt in MNAR_TARGETS:
        if tgt in p.columns:
            p[tgt] = (
                p.groupby("iso3c")[tgt]
                .transform(lambda s: s.ffill().bfill())
            )
    return _eval(p, truth_df, "ForwardFill")


def method_linear_interp(
    panel_amp: pd.DataFrame, truth_df: pd.DataFrame
) -> dict:
    """Linear interpolation within country, all gaps."""
    p = panel_amp.copy()
    for tgt in MNAR_TARGETS:
        if tgt in p.columns:
            p[tgt] = (
                p.groupby("iso3c")[tgt]
                .transform(lambda s: s.interpolate(
                    method="linear", limit_direction="both",
                    limit_area=None,
                ))
            )
    return _eval(p, truth_df, "LinearInterp")


def method_knn(panel_amp: pd.DataFrame, truth_df: pd.DataFrame) -> dict:
    """
    KNN imputation (k=5) using all available indicators as context.
    Applied to MNAR targets + covariates jointly.
    """
    p     = panel_amp.copy()
    cols  = MNAR_TARGETS + [c for c in IMPUTE_COVARIATES if c in p.columns]
    X     = p[cols].values.astype(float)
    imp   = KNNImputer(n_neighbors=5, weights="distance")
    X_imp = imp.fit_transform(X)
    for i, col in enumerate(cols):
        p[col] = X_imp[:, i]
    return _eval(p, truth_df, "KNN_k5")


def method_mice(panel_amp: pd.DataFrame, truth_df: pd.DataFrame) -> dict:
    """
    MICE (Multiple Imputation by Chained Equations) via IterativeImputer.
    Uses BayesianRidge as the internal estimator.
    Max 10 iterations, convergence tolerance 1e-3.
    """
    p     = panel_amp.copy()
    cols  = MNAR_TARGETS + [c for c in IMPUTE_COVARIATES if c in p.columns]
    X     = p[cols].values.astype(float)
    imp   = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        tol=1e-3,
        random_state=RANDOM_SEED,
        verbose=0,
    )
    X_imp = imp.fit_transform(X)
    for i, col in enumerate(cols):
        p[col] = X_imp[:, i]
    return _eval(p, truth_df, "MICE")


def method_income_group_mean(
    panel_amp: pd.DataFrame, truth_df: pd.DataFrame
) -> dict:
    """
    Income-group conditional mean imputation.
    For each missing cell, impute the mean of observed values
    within the same World Bank income tier.
    Specifically designed for the structural MNAR condition.
    """
    p = panel_amp.copy()
    for tgt in MNAR_TARGETS:
        if tgt not in p.columns:
            continue
        group_means = (
            p.groupby("income_group")[tgt]
            .transform("mean")
        )
        p[tgt] = p[tgt].fillna(group_means)
        # Fall back to global mean for any remaining NaN
        p[tgt] = p[tgt].fillna(p[tgt].mean())
    return _eval(p, truth_df, "IncomeGroupMean")


def method_gdp_regression(
    panel_amp: pd.DataFrame, truth_df: pd.DataFrame
) -> dict:
    """
    GDP-anchored regression imputation.
    For each MNAR target:
      1. Fit OLS: target ~ log(GDP/capita) on observed rows.
      2. Impute missing cells with fitted values.

    This exploits the empirical Kuznets-curve relationship
    between income level and inequality — countries with
    higher GDP/capita tend to have lower Gini.
    """
    p = panel_amp.copy()
    if "gdp_pc_ppp" not in p.columns:
        return _eval(p, truth_df, "GDPRegression")

    # Log-transform GDP
    log_gdp = np.log(p["gdp_pc_ppp"].clip(lower=1)).values.reshape(-1, 1)

    for tgt in MNAR_TARGETS:
        if tgt not in p.columns:
            continue
        observed_mask = p[tgt].notna() & p["gdp_pc_ppp"].notna()
        if observed_mask.sum() < 5:
            continue
        lr = LinearRegression()
        lr.fit(log_gdp[observed_mask], p.loc[observed_mask, tgt].values)
        fitted = lr.predict(log_gdp)
        p[tgt] = p[tgt].fillna(pd.Series(fitted, index=p.index))

    return _eval(p, truth_df, "GDPRegression")


# ──────────────────────────────────────────────────────────────
# Step 4 — Also impute the STRUCTURAL gap countries
# ──────────────────────────────────────────────────────────────
def impute_structural_gap(panel: pd.DataFrame) -> pd.DataFrame:
    """
    For countries with ALL Gini values missing (BDI, RWA, BGD),
    apply IncomeGroupMean and GDPRegression to produce filled panel.
    Returns the filled panel (used for downstream tasks E).
    Saves to data/processed/panel_12countries_imputed.csv.
    """
    p = panel.copy()

    # Income-group mean for MNAR targets
    for tgt in MNAR_TARGETS:
        if tgt not in p.columns:
            continue
        group_means = p.groupby("income_group")[tgt].transform("mean")
        p[tgt] = p[tgt].fillna(group_means)
        p[tgt] = p[tgt].fillna(p[tgt].mean())

    # GDP regression as cross-check
    if "gdp_pc_ppp" in p.columns:
        log_gdp = np.log(p["gdp_pc_ppp"].clip(lower=1)).values.reshape(-1,1)
        for tgt in MNAR_TARGETS:
            if tgt not in p.columns:
                continue
            obs_mask = p[tgt].notna() & p["gdp_pc_ppp"].notna()
            if obs_mask.sum() >= 5:
                lr = LinearRegression()
                lr.fit(log_gdp[obs_mask], p.loc[obs_mask, tgt].values)
                fitted = lr.predict(log_gdp)
                still_missing = p[tgt].isna()
                p.loc[still_missing, tgt] = fitted[still_missing]

    # Save imputed panel
    out = DATA_DIR / f"panel_12countries_imputed_{PERIOD_TAG}.csv"
    p.to_csv(out, index=False)
    n_filled = panel[MNAR_TARGETS].isna().sum().sum() - p[MNAR_TARGETS].isna().sum().sum()
    print(f"\n[step 4] Structural gap imputation: {n_filled} cells filled")
    print(f"  Saved: {out.name}")

    # Report per country
    for iso, grp in p.groupby("iso3c"):
        orig_miss = panel[panel["iso3c"]==iso][MNAR_TARGETS].isna().sum().sum()
        new_miss  = grp[MNAR_TARGETS].isna().sum().sum()
        if orig_miss > 0:
            tier = grp["income_group"].iloc[0]
            print(f"  {iso:<5} {tier:<25}  "
                  f"was {orig_miss} missing → now {new_miss} missing")
    return p


# ──────────────────────────────────────────────────────────────
# Figures
# ──────────────────────────────────────────────────────────────
def plot_mnar_heatmap(panel: pd.DataFrame) -> None:
    """
    Heatmap: rows = countries (sorted by income tier),
    cols = years, cell colour = Gini observed/missing.
    """
    countries_ordered = (
        panel.groupby("iso3c")["income_group"].first()
        .reset_index()
        .assign(_s=lambda d: d["income_group"].map(INCOME_ORDER).fillna(4))
        .sort_values("_s")["iso3c"].tolist()
    )
    iso_to_name = dict(zip(panel["iso3c"], panel["country"]))
    iso_to_tier = dict(zip(panel["iso3c"], panel["income_group"]))
    years = sorted(panel["year"].unique())

    TIER_COLOURS = {
        "High income":         "#4393c3",
        "Upper middle income": "#74c476",
        "Lower middle income": "#fd8d3c",
        "Low income":          "#d73027",
    }

    grid = np.zeros((len(countries_ordered), len(years)))
    for i, iso in enumerate(countries_ordered):
        for j, yr in enumerate(years):
            row = panel[(panel["iso3c"]==iso) & (panel["year"]==yr)]
            if not row.empty and pd.notna(row["gini"].iloc[0]):
                grid[i, j] = 1.0

    fig, ax = plt.subplots(figsize=(12, 6))
    cmap = matplotlib.colors.ListedColormap(["#f5f5f5", "#2166ac"])
    ax.imshow(grid, aspect="auto", interpolation="nearest",
              vmin=0, vmax=1, cmap=cmap)

    ax.set_xticks(range(len(years)))
    ax.set_xticklabels([str(y) for y in years], rotation=45,
                       ha="right", fontsize=9)
    ax.set_yticks(range(len(countries_ordered)))
    ytick_labels = [
        f"{iso} ({iso_to_name.get(iso,'')[:12]})"
        for iso in countries_ordered
    ]
    ax.set_yticklabels(ytick_labels, fontsize=9)
    for i, iso in enumerate(countries_ordered):
        tier = iso_to_tier.get(iso, "")
        ax.get_yticklabels()[i].set_color(TIER_COLOURS.get(tier, "black"))

    legend_handles = [
        mpatches.Patch(color="#2166ac", label="Observed"),
        mpatches.Patch(color="#f5f5f5", label="Missing (MNAR)",
                       edgecolor="#cccccc"),
    ] + [mpatches.Patch(color=c, label=t)
         for t, c in TIER_COLOURS.items()]

    ax.legend(handles=legend_handles, loc="upper left",
              bbox_to_anchor=(1.01, 1.0), fontsize=8,
              frameon=True, edgecolor="#cccccc")
    ax.set_title(
        f"Task D — Gini index: observed vs missing (MNAR pattern)\n"
        f"Income-gradient missingness: low-income countries "
        f"systematically lack Gini data",
        fontsize=10, fontweight="bold",
    )
    fig.tight_layout()
    out = FIGDIR / f"taskD_mnar_pattern_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_imputation_rmse(df: pd.DataFrame) -> None:
    """
    Grouped bar chart: RMSE per imputation method × indicator.
    """
    methods  = df["method"].tolist()
    x        = np.arange(len(methods))
    width    = 0.25
    tgt_colours = {
        "gini":           "#d73027",
        "inc_share_low20":"#4393c3",
        "poverty_215":    "#74c476",
    }
    tgt_labels = {
        "gini":           "Gini index",
        "inc_share_low20":"Income share (low 20%)",
        "poverty_215":    "Poverty ($2.15/day)",
    }

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, tgt in enumerate(MNAR_TARGETS):
        col   = f"rmse_{tgt}"
        vals  = df[col].values if col in df.columns else np.full(len(df), np.nan)
        offs  = (i - 1) * width
        bars  = ax.bar(x + offs, vals, width,
                       label=tgt_labels[tgt],
                       color=tgt_colours[tgt],
                       alpha=0.85, edgecolor="white")
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h) and h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.1,
                        f"{h:.2f}", ha="center", va="bottom",
                        fontsize=6.5, rotation=45)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel("RMSE", fontsize=10)
    ax.set_title(
        "Task D — Imputation RMSE by method and indicator\n"
        f"(amputation fraction = {AMPUTATE_FRAC:.0%}, "
        f"seed = {RANDOM_SEED})",
        fontsize=10, fontweight="bold",
    )
    ax.legend(fontsize=9, frameon=True, edgecolor="#cccccc")
    ax.grid(axis="y", alpha=0.25, lw=0.6)
    ax.spines[["top","right"]].set_visible(False)
    fig.tight_layout()
    out = FIGDIR / f"taskD_imputation_rmse_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


def plot_imputed_gini(panel: pd.DataFrame, panel_imp: pd.DataFrame) -> None:
    """
    Line plot: original vs imputed Gini trajectory for countries
    with structural MNAR gaps, using best imputation method.
    """
    mnar_isos = [iso for iso, grp in panel.groupby("iso3c")
                 if grp["gini"].isna().all()]

    if not mnar_isos:
        print("  [skip] No all-missing Gini countries — skipping imputed plot")
        return

    fig, axes = plt.subplots(
        1, len(mnar_isos),
        figsize=(5 * len(mnar_isos), 4),
        sharey=False,
        gridspec_kw={"wspace": 0.35},
    )
    if len(mnar_isos) == 1:
        axes = [axes]

    TIER_COLOURS = {
        "High income":         "#4393c3",
        "Upper middle income": "#74c476",
        "Lower middle income": "#fd8d3c",
        "Low income":          "#d73027",
    }

    for ax, iso in zip(axes, mnar_isos):
        grp_orig = panel[panel["iso3c"] == iso].sort_values("year")
        grp_imp  = panel_imp[panel_imp["iso3c"] == iso].sort_values("year")
        tier     = grp_orig["income_group"].iloc[0]
        name     = grp_orig["country"].iloc[0]
        colour   = TIER_COLOURS.get(tier, "grey")

        # Imputed series
        ax.plot(grp_imp["year"], grp_imp["gini"],
                color=colour, lw=2.0, ls="--", marker="o", markersize=5,
                label="Imputed (income-group mean)", zorder=3)

        # Reference band: ±5pp uncertainty
        ax.fill_between(
            grp_imp["year"],
            grp_imp["gini"] - 5, grp_imp["gini"] + 5,
            color=colour, alpha=0.12, label="±5 pp uncertainty band",
        )

        # Regional median from observed countries
        tier_obs = (panel[panel["income_group"] == tier]
                    .groupby("year")["gini"]
                    .median())
        ax.plot(tier_obs.index, tier_obs.values,
                color="#888888", lw=1.2, ls=":", label="Tier median (observed)")

        ax.set_title(f"{iso} ({name})\n{tier}", fontsize=9, fontweight="bold")
        ax.set_xlabel("Year", fontsize=9)
        ax.set_ylabel("Gini index", fontsize=9)
        ax.set_ylim(20, 65)
        ax.grid(alpha=0.2, lw=0.5)
        ax.spines[["top","right"]].set_visible(False)
        ax.legend(fontsize=7, frameon=True, edgecolor="#cccccc")

    fig.suptitle(
        "Task D — Imputed Gini trajectories for MNAR countries\n"
        "(all Gini values originally missing; imputed via income-group mean + GDP regression)",
        fontsize=10, fontweight="bold", y=1.01,
    )
    fig.tight_layout()
    out = FIGDIR / f"taskD_imputed_gini_{PERIOD_TAG}.png"
    fig.savefig(out, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  [OK] {out.name}")


# ──────────────────────────────────────────────────────────────
# Save outputs
# ──────────────────────────────────────────────────────────────
def save_results(df: pd.DataFrame) -> None:
    print("\n[step 5] Saving results …")

    df.to_csv(RESULT_DIR / "task_D_metrics.csv",  index=False)
    df.to_csv(TABLE_DIR  / "table_taskD.csv",      index=False)
    print(f"  [OK] task_D_metrics.csv")

    # LaTeX
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Task D --- MNAR imputation benchmark. "
        r"RMSE evaluated on $" + str(int(AMPUTATE_FRAC*100)) +
        r"\%$ of observed Gini, income-share, and poverty values "
        r"masked at random (seed=" + str(RANDOM_SEED) + r"). "
        r"Income-Group Mean and GDP Regression target the structural "
        r"MNAR gap (countries with all values missing).}",
        r"\label{tab:taskD}",
        r"\begin{tabular}{lrrrrrr}",
        r"\toprule",
        r"Method & RMSE$_{\text{all}}$ & MAE$_{\text{all}}$ & "
        r"RMSE$_{\text{Gini}}$ & RMSE$_{\text{Inc}}$ & "
        r"RMSE$_{\text{Pov}}$ & $n$ \\",
        r"\midrule",
    ]
    def f(v): return f"{v:.3f}" if pd.notna(v) else "--"
    for _, row in df.iterrows():
        name = row["method"].replace("_", r"\_")
        lines.append(
            f"{name} & {f(row['rmse_overall'])} & {f(row['mae_overall'])} & "
            f"{f(row.get('rmse_gini', np.nan))} & "
            f"{f(row.get('rmse_inc_share_low20', np.nan))} & "
            f"{f(row.get('rmse_poverty_215', np.nan))} & "
            f"{int(row['n_evaluated'])} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    (TABLE_DIR / "table_taskD.tex").write_text(
        "\n".join(lines), encoding="utf-8"
    )
    print(f"  [OK] table_taskD.csv / .tex")
    print(f"\n[done] Results : {RESULT_DIR.resolve()}")
    print(f"[done] Tables  : {TABLE_DIR.resolve()}")
    print(f"[done] Figures : {FIGDIR.resolve()}")
    print("\n[next] Run: python src/09_task_E_causal_ml.py")


def print_summary(df: pd.DataFrame) -> None:
    print(f"\n{'='*68}")
    print("Task D — Imputation results")
    print(f"{'='*68}")
    print(f"  {'Method':<22}  {'RMSE_all':>9}  {'MAE_all':>8}  "
          f"{'RMSE_gini':>10}  {'n':>4}")
    print(f"  {'-'*22}  {'-'*9}  {'-'*8}  {'-'*10}  {'-'*4}")
    for _, row in df.iterrows():
        def f(v, w): return f"{v:{w}.3f}" if pd.notna(v) else f"{'--':>{w}}"
        print(f"  {row['method']:<22}  {f(row['rmse_overall'],9)}  "
              f"{f(row['mae_overall'],8)}  "
              f"{f(row.get('rmse_gini', np.nan),10)}  "
              f"{int(row['n_evaluated']):>4}")
    print(f"{'='*68}")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    panel, mask = load_data()

    mnar_df = describe_mnar(panel)

    print("\n[step 2] Amputation experiment …")
    panel_amp, truth_df = ampute(panel)

    print("\n[step 3] Running imputation methods …")
    methods = [
        method_mean,
        method_median,
        method_ffill,
        method_linear_interp,
        method_knn,
        method_mice,
        method_income_group_mean,
        method_gdp_regression,
    ]
    results = []
    for fn in methods:
        r = fn(panel_amp, truth_df)
        results.append(r)
        print(f"  {r['method']:<22}  "
              f"RMSE={r['rmse_overall']:.3f}  "
              f"MAE={r['mae_overall']:.3f}  "
              f"n={r['n_evaluated']}")

    df = pd.DataFrame(results)
    print_summary(df)

    print("\n[step 4] Structural MNAR gap imputation …")
    panel_imp = impute_structural_gap(panel)

    print("\n[step 5] Figures …")
    plot_mnar_heatmap(panel)
    plot_imputation_rmse(df)
    plot_imputed_gini(panel, panel_imp)

    save_results(df)


if __name__ == "__main__":
    main()